## **Dependencies**

In [1]:
!apt-get update -qq
!apt-get install -y -qq ffmpeg tesseract-ocr poppler-utils

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)


## **Backages**

In [2]:
!pip install -q torch==2.6.0 torchvision==0.21.0 torchaudio==2.6.0 --index-url https://download.pytorch.org/whl/cu124
!pip install -q torchaudio==2.6.0 --index-url https://download.pytorch.org/whl/cu124
!pip install -q huggingface-hub==0.27.1 tokenizers==0.20.3
!pip install -q transformers==4.46.3 sentence-transformers==3.3.1



import torch

print("PyTorch:", torch.__version__)
print("GPU available:", torch.cuda.is_available())
print("GPU name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")

PyTorch: 2.6.0+cu124
GPU available: False
GPU name: None


In [3]:
!pip install -q requests beautifulsoup4
!pip install -q langchain langchain-community langchain-text-splitters
!pip install -q faiss-cpu
!pip install -q git+https://github.com/openai/whisper.git
!pip install -q PyPDF2 pdfplumber python-docx pytesseract
!pip install -q pillow==10.4.0
!pip install -q whispher
!pip install -q PyMuPDF
!pip install -q gtts
!pip install -q gradio==4.44.1

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio-client 1.3.0 requires websockets<13.0,>=10.0, but you have websockets 16.1.1 which is incompatible.
hf-gradio 0.4.1 requires gradio-client<3.0,>=2.0, but you have gradio-client 1.3.0 which is incompatible.
google-adk 2.7.1 requires websockets<16,>=15.0.1, but you have websockets 16.1.1 which is incompatible.
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 4.44.1 requires pillow<11.0,>=8.0, but you have pillow 12.3.0 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.27.1 which is incom

In [4]:
!pip install -q sounddevice scipy

## **Importing**

In [5]:
import requests
from bs4 import BeautifulSoup
import torch
import torchvision
import PIL
import whisper
import faiss
import pymupdf
from google.colab import files
from sentence_transformers import SentenceTransformer
from transformers import PreTrainedModel , pipeline

## **Functions**

### **Input functions**

#### handle_text

In [6]:
def handle_text(query):
    if not query or not query.strip():
        return {'answer': "Please provide a question.",
                'citations': [], 'audio': None}

    result = generate_with_citations(query.strip())

    return {
        'answer': result['answer'],
        'citations': result['citations'],
        'audio': None,
        'input_type': 'text'
    }

#### Handling_voice

In [7]:

from google.colab import output as colab_output
from IPython.display import display, HTML, Audio
import base64, os


def _on_recording_received(b64_data):
    path = '/content/recording.webm'

    with open(path, 'wb') as f:
        f.write(base64.b64decode(b64_data))
    print(" Recording received")

    try:
        r = handle_voice(path)
    except Exception as e:
        print(f" handle_voice failed: {e}")
        import traceback
        traceback.print_exc()
        return "error"

    print(f"\nYou said: {r.get('transcript','')}")
    print(f"Answer: {r.get('answer','')}")

    # 👇 display citations
    cites = r.get('citations') or []
    if cites:
        print("\n Citations:")
        for i, c in enumerate(cites, 1):
            if isinstance(c, dict):
                src   = c.get('source') or c.get('url') or c.get('title') or '?'
                title = c.get('title') or ''
                print(f"  [{i}] {title} — {src}")
            else:
                print(f"  [{i}] {c}")
    else:
        print("\n (no citations)")

    if r.get('audio') and os.path.exists(r['audio']):
        display(Audio(r['audio']))

    #main_menu()
    return "done"


colab_output.register_callback('notebook.handle_recording', _on_recording_received)


RECORDER_HTML = """
<div style="font-family: sans-serif; padding: 15px; border: 2px solid #4CAF50; border-radius: 10px; background: #f9f9f9; max-width: 520px;">
  <button id="recBtn" style="padding: 12px 24px; background: #d33; color: white; border: none; border-radius: 6px; font-size: 16px; cursor: pointer;">
     Start Recording
  </button>
  <span id="status" style="margin-left: 15px; font-size: 14px;">Idle</span>
  <br><br>
  <audio id="player" controls style="display:none; width: 100%;"></audio>
  <br>
  <button id="sendBtn" style="padding: 10px 20px; background: #28a; color: white; border: none; border-radius: 6px; cursor: pointer; display:none;">
    ⬆ Send to Colab
  </button>
</div>

<script>
let mediaRecorder, chunks = [], audioBlob;
const recBtn = document.getElementById('recBtn');
const status = document.getElementById('status');
const player = document.getElementById('player');
const sendBtn = document.getElementById('sendBtn');

recBtn.onclick = async () => {
  if (mediaRecorder && mediaRecorder.state === 'recording') {
    mediaRecorder.stop();
    recBtn.textContent = ' Start Recording';
    recBtn.style.background = '#d33';
    return;
  }
  try {
    const stream = await navigator.mediaDevices.getUserMedia({ audio: true });
    mediaRecorder = new MediaRecorder(stream);
    chunks = [];
    mediaRecorder.ondataavailable = e => { if (e.data.size > 0) chunks.push(e.data); };
    mediaRecorder.onstop = () => {
      audioBlob = new Blob(chunks, { type: 'audio/webm' });
      player.src = URL.createObjectURL(audioBlob);
      player.style.display = 'block';
      sendBtn.style.display = 'inline-block';
      status.textContent = ' Done. Click Send.';
      stream.getTracks().forEach(t => t.stop());
    };
    mediaRecorder.start();
    recBtn.textContent = 'Stop Recording';
    recBtn.style.background = '#333';
    status.textContent = ' Recording...';
  } catch (err) {
    status.textContent =  err.message;
  }
};

sendBtn.onclick = () => {
  if (!audioBlob) return;
  const reader = new FileReader();
  reader.onload = () => {
    const b64 = reader.result.split(',')[1];
    google.colab.kernel.invokeFunction('notebook.handle_recording', [b64], {});
    status.textContent = 'Sent! Answer appears below.';
  };
  reader.readAsDataURL(audioBlob);
};
</script>
"""


def record_voice():
    print(" Click 'Start Recording' → speak → Stop → 'Send to Colab'")
    print("   The answer will appear automatically.\n")
    display(HTML(RECORDER_HTML))

In [8]:
import whisper
import os
import subprocess
from gtts import gTTS


_whisper_model = None

def get_whisper():
    global _whisper_model
    if _whisper_model is None:
        print("🎙 Loading Whisper...")
        #_whisper_model = whisper.load_model("base")
        _whisper_model = whisper.load_model("medium")
        print("Whisper loaded")
    return _whisper_model


def convert_to_wav(input_path, output_path="/content/fixed_audio.wav"):
    if not os.path.exists(input_path):
        return None
    cmd = ["ffmpeg", "-y", "-i", input_path,
           "-ac", "1", "-ar", "16000", "-f", "wav", output_path]
    try:
        subprocess.run(cmd, check=True,
                       stdout=subprocess.DEVNULL,
                       stderr=subprocess.DEVNULL)
        return output_path
    except Exception as e:
        print(f"ffmpeg failed: {e}")
        return None


def voice_to_text(audio_path):
    if not audio_path or not os.path.exists(audio_path):
        return {'transcript': '', 'success': False}
    wav_path = convert_to_wav(audio_path)
    if not wav_path:
        return {'transcript': '', 'success': False}
    result = get_whisper().transcribe(wav_path, language=None)
    transcript = result['text'].strip()
    return {'transcript': transcript, 'success': bool(transcript)}


def text_to_speech(text, output_path="/content/answer.mp3"):
    lang = 'ar' if any('\u0600' <= c <= '\u06FF' for c in text) else 'en'
    tts = gTTS(text=text, lang=lang, slow=False)
    tts.save(output_path)
    return output_path


def handle_voice(audio_path):
    if not audio_path or not os.path.exists(audio_path):
        return {'transcript': '', 'answer': "Audio file not found.",
                'audio': None, 'input_type': 'voice', 'citations': []}

    v = voice_to_text(audio_path)
    if not v['success']:
        print(" voice_to_text failed (empty or untranscribable audio)")
        return {'transcript': '', 'answer': "Could not transcribe audio.",
                'audio': None, 'input_type': 'voice', 'citations': []}
    transcript = v['transcript']
    print(f" transcript: {transcript!r}")

    try:
        r = generate_with_citations(transcript)
    except Exception as e:
        print(f" generate_with_citations failed: {e}")
        return {'transcript': transcript,
                'answer': "Could not generate an answer.",
                'audio': None, 'input_type': 'voice', 'citations': []}

    answer    = r.get('answer', '')
    citations = r.get('citations') or r.get('sources') or r.get('sources_used') or []

    try:
        audio_out = text_to_speech(answer)
    except Exception as e:
        print(f" text_to_speech failed: {e}")
        audio_out = None

    return {
        'transcript': transcript,
        'answer': answer,
        'audio': audio_out,
        'input_type': 'voice',
        'citations': citations,
    }

#### Handling_docs

In [9]:
import os
import pymupdf
import docx
import pytesseract
from PIL import Image


def parse_pdf(path):
    doc = pymupdf.open(path)
    text = ""
    for page in doc:
        text += page.get_text() + "\n"
    doc.close()
    return text


def parse_docx(path):
    doc = docx.Document(path)
    return "\n".join(p.text for p in doc.paragraphs)


def parse_txt(path):
    with open(path, 'r', encoding='utf-8', errors='ignore') as f:
        return f.read()


def parse_image(path):
    img = Image.open(path)
    return pytesseract.image_to_string(img, lang='ara+eng')


def parse_document(path):

    if not path or not os.path.exists(path):
        raise FileNotFoundError(f"File not found: {path}")

    ext = path.lower().rsplit('.', 1)[-1]

    if ext == 'pdf':   return parse_pdf(path)
    if ext == 'docx':  return parse_docx(path)
    if ext == 'txt':   return parse_txt(path)
    if ext in ('png', 'jpg', 'jpeg'): return parse_image(path)

    raise ValueError(f"Unsupported file type: .{ext}")



In [10]:
import os

def doc_ingest(path):
    text = parse_document(path)
    if not text.strip():
        return {'answer': " No text extracted.",
                'citations': [], 'input_type': 'document'}

    chunks = chunk_text(text, metadata={
        'source': path,
        'title': os.path.basename(path),
        'type': 'user_document'
    })

    n = add_to_index(chunks)
    return {
        'answer': f" Document ingested: {n} chunks added to knowledge base.",
        'citations': [],
        'input_type': 'document'
    }


def doc_ingest_and_query(path, question):
    text = parse_document(path)
    if not text.strip():
        return {'answer': "No text extracted.",
                'citations': [], 'input_type': 'document'}

    chunks = chunk_text(text, metadata={
        'source': path,
        'title': os.path.basename(path),
        'type': 'user_document'
    })

    add_to_index(chunks)

    r = generate_with_citations(question)

    return {
        'answer': r['answer'],
        'citations': r.get('citations', []),
        'input_type': 'document'
    }


def doc_query_only(path, question):
    text = parse_document(path)
    if not text.strip():
        return {'answer': "No text extracted.",
                'citations': [], 'input_type': 'document'}

    chunks = chunk_text(text, metadata={
        'source': path,
        'title': os.path.basename(path),
        'type': 'temp'
    })

    results = search_in_temp_index(chunks, question)
    r = generate_from_chunks(results, question)

    return {
        'answer': r['answer'],
        'citations': r.get('citations', []),
        'input_type': 'document'
    }


In [11]:
def search_in_temp_index(chunks, question):
    q_emb = embed_texts([question]).astype('float32')
    c_embs = embed_texts([c['text'] for c in chunks]).astype('float32')
    temp_index = faiss.IndexFlatL2(c_embs.shape[1])
    temp_index.add(c_embs)
    distances, indices = temp_index.search(q_emb, min(3, len(chunks)))
    return [{**chunks[idx], 'distance': float(distances[0][i])}
            for i, idx in enumerate(indices[0]) if idx >= 0]

def generate_from_chunks(results, question):
    context = "\n".join([r['text'][:300] for r in results])
    messages = [
        {"role": "system", "content": "Answer using ONLY the context."},
        {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}"}
    ]
    output = llm(messages, max_new_tokens=200)
    answer = output[0]['generated_text'][-1]['content'].strip()
    citations = [{'id': i+1, 'title': r['title'], 'url': r['source'], 'type': r['type']}
                 for i, r in enumerate(results)]
    return {'answer': answer, 'citations': citations}

In [12]:
def ingest_document(path):
    text = parse_document(path)
    if not text.strip():
        return 0
    chunks = chunk_text(text, metadata={
        'source': path,
        'title': path.split('/')[-1],
        'type': 'user_document'
    })
    add_to_index(chunks)
    return len(chunks)

### Preprocessing_functions

### Chunck_function

In [13]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

_text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    separators=["\n\n", "\n", ".", " ", ""]
)

def chunk_text(text, metadata=None):
    meta = metadata or {}
    pieces = _text_splitter.split_text(text)
    return [
        {
            'text': p.strip(),
            'source': meta.get('source', 'unknown'),
            'title': meta.get('title', ''),
            'type': meta.get('type', 'text')
        }
        for p in pieces if len(p.strip()) > 50
    ]

### embed_funcction

In [14]:
from sentence_transformers import SentenceTransformer

_embed_model = None

def get_embed_model():
    global _embed_model
    if _embed_model is None:
        print("Loading embedding model...")
        _embed_model = SentenceTransformer('BAAI/bge-m3')
        print("Model loaded")
    return _embed_model

def embed_texts(texts):
    return get_embed_model().encode(texts, show_progress_bar=False, batch_size=32)



### faiss_functions

In [15]:
import faiss
import pickle
import numpy as np

_index = None
_chunks = []

def create_index(dimension=1024):
    global _index, _chunks
    _index = faiss.IndexFlatL2(dimension)
    _chunks = []
    print(f"Index created (dim={dimension})")

def add_to_index(chunks):
    global _index, _chunks
    if not chunks:
        return 0
    texts = [c['text'] for c in chunks]
    embeddings = embed_texts(texts).astype('float32')
    if _index is None:
        create_index(embeddings.shape[1])
    _index.add(embeddings)
    _chunks.extend(chunks)
    return len(chunks)

def search_index(query, top_k=3):
    if _index is None or _index.ntotal == 0:
        return []
    query_emb = embed_texts([query]).astype('float32')
    distances, indices = _index.search(query_emb, top_k)
    results = []
    for i, idx in enumerate(indices[0]):
        if idx >= 0:
            results.append({**_chunks[idx], 'distance': float(distances[0][i])})
    return results

def save_index(index_path='index.faiss', chunks_path='chunks.pkl'):
    faiss.write_index(_index, index_path)
    with open(chunks_path, 'wb') as f:
        pickle.dump(_chunks, f)
    print(f"Saved: {index_path}, {chunks_path}")

def load_index(index_path='index.faiss', chunks_path='chunks.pkl'):
    global _index, _chunks
    _index = faiss.read_index(index_path)
    with open(chunks_path, 'rb') as f:
        _chunks = pickle.load(f)
    print(f" Loaded: {_index.ntotal} vectors, {len(_chunks)} chunks")

### Web Scrapping

In [16]:
def scrape_page(url):
    headers = {'User-Agent': 'Mozilla/5.0'}
    response = requests.get(url, headers=headers, timeout=10)
    soup = BeautifulSoup(response.text, 'html.parser')

    for tag in soup(['script', 'style', 'nav', 'footer']):
        tag.decompose()

    text = soup.get_text(separator='\n', strip=True)
    lines = [l.strip() for l in text.split('\n') if l.strip()]

    return {
        'url': url,
        'title': soup.title.string if soup.title else '',
        'content': '\n'.join(lines)
    }


In [17]:
result = scrape_page("https://te.eg")
print(f"Title: {result['title']}")
print(f"Content length: {len(result['content'])} characters")
print(f"\nFirst 500 chars:\n{result['content'][:500]}")

Title: WE مصر | باقات الإنترنت والموبايل وخدمات الاتصالات - Telecom Egypt
Content length: 1568 characters

First 500 chars:
WE مصر | باقات الإنترنت والموبايل وخدمات الاتصالات - Telecom Egypt
Warning
تنبيه
×
Products in comparison will be cleared
سيتم محو المنتجات التي يتم مقارنتها
Cancel
إلغاء
Ok
تم
Previous
Next
احجز رقم WE Gold
احجز رقم WE Gold
جدد إشتراك WE إنترنت
جدد إشتراك WE إنترنت
دفع فاتورة التليفون الأرضى
دفع فاتورة التليفون الأرضى
شحن / دفع فاتورة الموبايل
شحن / دفع فاتورة الموبايل
خدمات مميزة
الأسئلة الشائعة eKYC
اعرف عميلك الكترونياً eKYC هو عملية رقمية تتيح للشركة التحقق من هويتك بشكل آمن باستخدام هاتفك 


#### whole webScraping across the website for all pages .

In [18]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse

def get_all_links(start_url, max_pages=100):
    visited = set()
    to_visit = [start_url]
    all_links = set()

    headers = {'User-Agent': 'Mozilla/5.0'}

    while to_visit and len(visited) < max_pages:
        url = to_visit.pop(0)
        if url in visited:
            continue

        try:
            response = requests.get(url, headers=headers, timeout=10)
            soup = BeautifulSoup(response.text, 'html.parser')
            visited.add(url)

            for a in soup.find_all('a', href=True):
                link = urljoin(url, a['href'])
                parsed = urlparse(link)

                if 'te.eg' in parsed.netloc:
                    clean_link = f"{parsed.scheme}://{parsed.netloc}{parsed.path}"
                    all_links.add(clean_link)
                    if clean_link not in visited:
                        to_visit.append(clean_link)

        except Exception as e:
            print(f"Error: {url} - {e}")

    return all_links

links = get_all_links("https://te.eg", max_pages=50)
print(f"Found {len(links)} unique pages")
for link in sorted(links)[:30]:
    print(link)

Found 420 unique pages
http://www.te.eg/app
https://arabicdomains.te.eg/ar/TEData-Business/Arabic-Domains-Subscribe
https://beta.te.eg/ar/personal/devices/4g-routers
https://csr.te.eg/ar
https://csr.te.eg/en
https://ir.te.eg/
https://ir.te.eg/ar
https://my.te.eg/echannel/
https://my.te.eg/user/login
https://numbers.te.eg/echannelb2b/
https://shop.te.eg/echannel/
https://te.eg
https://te.eg/Corporate-Sustainability/
https://te.eg/Personal/Devices/4G-Routers/
https://te.eg/Store-Locator
https://te.eg/about-te/
https://te.eg/about-te/Ma3ak
https://te.eg/about-te/board-of-directors
https://te.eg/about-te/careers-and-training
https://te.eg/about-te/contact-us
https://te.eg/about-te/faq
https://te.eg/about-te/live-chat
https://te.eg/about-te/ma3ak
https://te.eg/about-te/management-team
https://te.eg/ar/personal/ekyc-faqs
https://te.eg/ar/personal/home/one-home/600min/30mbps
https://te.eg/ar/web/guest/w/control-double-bundle-offer
https://te.eg/ar/web/guest/w/devices-summer26
https://te.eg/ar

In [19]:
import time
import json

def scrape_page(url):
    headers = {'User-Agent': 'Mozilla/5.0'}
    try:
        response = requests.get(url, headers=headers, timeout=10)
        soup = BeautifulSoup(response.text, 'html.parser')

        for tag in soup(['script', 'style', 'nav', 'footer']):
            tag.decompose()

        text = soup.get_text(separator='\n', strip=True)
        lines = [l.strip() for l in text.split('\n') if l.strip()]

        return {
            'url': url,
            'title': soup.title.string if soup.title else '',
            'content': '\n'.join(lines)
        }
    except Exception as e:
        print(f" Failed: {url} - {e}")
        return None

# Scrape all pages
scraped_data = []
for i, url in enumerate(sorted(links)):
    print(f"[{i+1}/{len(links)}] Scraping: {url}")
    result = scrape_page(url)
    if result and len(result['content']) > 100:  # Skip near-empty pages
        scraped_data.append(result)
    time.sleep(0.5)  # Be polite to the server

print(f"\n Scraped {len(scraped_data)} pages successfully")

[1/420] Scraping: http://www.te.eg/app
 Failed: http://www.te.eg/app - HTTPConnectionPool(host='www.te.eg', port=80): Max retries exceeded with url: /app (Caused by ConnectTimeoutError(<urllib3.connection.HTTPConnection object at 0x7e35550a5be0>, 'Connection to www.te.eg timed out. (connect timeout=10)'))
[2/420] Scraping: https://arabicdomains.te.eg/ar/TEData-Business/Arabic-Domains-Subscribe
 Failed: https://arabicdomains.te.eg/ar/TEData-Business/Arabic-Domains-Subscribe - HTTPSConnectionPool(host='arabicdomains.te.eg', port=443): Max retries exceeded with url: /ar/TEData-Business/Arabic-Domains-Subscribe (Caused by ConnectTimeoutError(<urllib3.connection.HTTPSConnection object at 0x7e3554fcbed0>, 'Connection to arabicdomains.te.eg timed out. (connect timeout=10)'))
[3/420] Scraping: https://beta.te.eg/ar/personal/devices/4g-routers
 Failed: https://beta.te.eg/ar/personal/devices/4g-routers - HTTPSConnectionPool(host='beta.te.eg', port=443): Max retries exceeded with url: /ar/persona

KeyboardInterrupt: 

#### Faiss Database building [search]

In [22]:
import json

with open('te_eg_data.json', 'w', encoding='utf-8') as f:
    json.dump(scraped_data, f, ensure_ascii=False, indent=2)

print(f"Saved {len(scraped_data)} pages to te_eg_data.json")

# Show total content size
total_chars = sum(len(p['content']) for p in scraped_data)
print(f"Total content: {total_chars:,} characters")

Saved 26 pages to te_eg_data.json
Total content: 127,331 characters


In [23]:
import json

with open('te_eg_data.json', 'r', encoding='utf-8') as f:
    scraped_data = json.load(f)

print(f"Loaded {len(scraped_data)} pages\n")

create_index()

total = 0
for page in scraped_data:
    chunks = chunk_text(
        page['content'],
        metadata={
            'source': page['url'],
            'title': page['title'],
            'type': 'website'
        }
    )
    added = add_to_index(chunks)
    total += added

print(f"\n Baseline complete: {total} chunks")
print(f" Total vectors in FAISS: {_index.ntotal}")

save_index('baseline_index.faiss', 'baseline_chunks.pkl')

Loaded 26 pages

Index created (dim=1024)
Loading embedding model...
Model loaded

 Baseline complete: 338 chunks
 Total vectors in FAISS: 338
Saved: baseline_index.faiss, baseline_chunks.pkl


#### search_index_test

In [24]:
results = search_index("What internet packages does Telecom Egypt offer?", top_k=3)

for r in results:
    print(f"\n📄 {r['title']} ({r['type']})")
    print(f"   Source: {r['source']}")
    print(f"   Text: {r['text'][:200]}...")


📄 Telecom Egypt  :: Investor Relations (website)
   Source: https://ir.te.eg/
   Text: Read More
About Us
Telecom Egypt is the first total telecom operator in Egypt providing all telecom services to its customers including fixed and mobile voice and data services. Telecom Egypt has a lo...

📄 Telecom Egypt  :: Investor Relations (website)
   Source: https://ir.te.eg/
   Text: Financial Statements
Earnings Releases
Telecom Egypt to Advance Egypt’s Eastern Digital Corridor with a Multiple-Petabit Sharm El Sheikh–Taba Subsea Cable in Accordance with its New Strategy
03 Septem...

📄 WE مصر | باقات الإنترنت والموبايل وخدمات الاتصالات - Telecom Egypt (website)
   Source: https://te.eg
   Text: WE مصر | باقات الإنترنت والموبايل وخدمات الاتصالات - Telecom Egypt
Warning
تنبيه
×
Products in comparison will be cleared
سيتم محو المنتجات التي يتم مقارنتها
Cancel
إلغاء
Ok
تم
Previous
Next
احجز رقم ...


### LMM Generation

In [21]:
from transformers import pipeline

print("Loading Qwen...")
llm = pipeline(
    "text-generation",
    model="Qwen/Qwen2.5-0.5B-Instruct",
    device=-1,
    max_new_tokens=200,
    do_sample=False,
    temperature=None,
    top_p=None,
    top_k=None
)
print("✅ Qwen loaded (CPU)")

Loading Qwen...


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


✅ Qwen loaded (CPU)


### Rag Architecture

In [25]:
def generate_with_citations(query, top_k=3):
    """Full RAG with strong grounding + citation dedup + language matching."""

    # ─── STEP 1: RETRIEVE ───
    results = search_index(query, top_k=top_k + 2)  # Get extra for filtering

    # ─── STEP 2: FILTER NOISY SOURCES ───
    results = [r for r in results if 'ir.te.eg' not in r.get('source', '')]

    # ─── STEP 3: DEDUPLICATE BY SOURCE ───
    seen_urls = set()
    filtered = []
    for r in results:
        if r['source'] not in seen_urls:
            filtered.append(r)
            seen_urls.add(r['source'])
        if len(filtered) >= top_k:
            break
    results = filtered

    if not results:
        return {
            "answer": "I don't have information about that in my knowledge base.",
            "citations": []
        }

    context = "\n\n".join([
        f"[{i+1}] {r['text'][:400]}"
        for i, r in enumerate(results)
    ])

    is_arabic = any('\u0600' <= c <= '\u06FF' for c in query)

    if is_arabic:
        sys_prompt = (
            "أنت مساعد خدمة عملاء لشركة Telecom Egypt. "
            "أجب فقط بالاعتماد على السياق المقدم. "
            "إذا لم يكن الجواب موجوداً في السياق، قل: 'لا أملك هذه المعلومة'. "
            "لا تخترع أية معلومات. استشهد بالمصادر باستخدام [1]، [2]."
        )
    else:
        sys_prompt = (
            "You are a Telecom Egypt customer service assistant. "
            "Answer ONLY using the context provided. "
            "If the answer is not in the context, say: 'I don't have that information.' "
            "Do NOT make up information. Do NOT create lists unless the context has a list. "
            "Cite sources using [1], [2] format. Be concise (2-3 sentences)."
        )

    messages = [
        {"role": "system", "content": sys_prompt},
        {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {query}\n\nAnswer:"}
    ]

    output = llm(messages, max_new_tokens=200)
    answer = output[0]['generated_text'][-1]['content'].strip()

    citations = [
        {
            'id': i + 1,
            'title': r['title'],
            'url': r['source'],
            'type': r['type'],
            'snippet': r['text'][:150] + "..."
        }
        for i, r in enumerate(results)
    ]

    return {'answer': answer, 'citations': citations}

In [26]:
load_index('baseline_index.faiss', 'baseline_chunks.pkl')

 Loaded: 338 vectors, 338 chunks


#### test_rag

In [27]:
result = generate_with_citations("What internet packages does Telecom Egypt offer?")

print("=" * 60)
print("ANSWER:")
print("=" * 60)
print(result['answer'])

print("\n" + "=" * 60)
print("SOURCES:")
print("=" * 60)
for c in result['citations']:
    print(f"\n[{c['id']}] {c['title']}")
    print(f"    URL: {c['url']}")

ANSWER:
Telecom Egypt offers several internet packages including:

1. WE Gold
2. WE Platinum
3. WE Silver
4. WE Diamond
5. One Home Plan

These packages provide different levels of internet speed and features based on your budget and needs.

SOURCES:

[1] WE مصر | باقات الإنترنت والموبايل وخدمات الاتصالات - Telecom Egypt
    URL: https://te.eg

[2] باقاتOneHome | خط أرضى مع إنترنت منزلى - Telecom Egypt
    URL: https://te.eg/ar/personal/home/one-home/600min/30mbps


In [28]:

def process_input(user_input=None, input_type=None, doc_action=None, question=None):

    # ─── AUTO-DETECT if not given ───
    if input_type is None:
        if user_input is None:
            return {'answer': "No input provided.", 'input_type': 'unknown'}

        if not isinstance(user_input, str):
            return {'answer': "Invalid input type.", 'input_type': 'unknown'}

        ext = user_input.lower().rsplit('.', 1)[-1] if '.' in user_input else ''

        if ext in ('wav', 'mp3', 'm4a', 'webm', 'ogg'):
            input_type = 'voice'
        elif ext in ('pdf', 'docx', 'txt', 'png', 'jpg', 'jpeg'):
            input_type = 'document'
        else:
            input_type = 'text'

    if input_type == 'text':
        return handle_text(user_input)

    elif input_type == 'voice':
        return handle_voice(user_input)

    elif input_type == 'document':
        action = doc_action or 'ingest'
        if action == 'ingest':
            return doc_ingest(user_input)
        elif action == 'ingest_query':
            return doc_ingest_and_query(user_input, question or "")
        elif action == 'query':
            return doc_query_only(user_input, question or "")
        else:
            return {'answer': f"Unknown doc action: {action}", 'input_type': 'document'}

    else:
        return {'answer': f"Unknown input type: {input_type}", 'input_type': 'unknown'}


print("✅ process_input() ready")



def show_result(r):
    print("\n" + "=" * 60)

    if not r or not isinstance(r, dict):
        print(" No result.")
        print("=" * 60)
        return

    if r.get('transcript'):
        print(f" You said: {r['transcript']}")

    print(f" Answer: {r.get('answer', '(no answer)')}")

    if r.get('citations'):
        print("\ Sources:")
        for c in r['citations']:
            print(f"  [{c['id']}] {c['title']} → {c['url']}")

    if r.get('audio'):
        print("\n Response audio:")
        display(Audio(r['audio']))

    print("=" * 60)


print(" Handlers ready: handle_text, handle_voice, doc_ingest, doc_ingest_and_query, doc_query_only")

✅ process_input() ready
 Handlers ready: handle_text, handle_voice, doc_ingest, doc_ingest_and_query, doc_query_only


<>:59: SyntaxWarning: invalid escape sequence '\ '
<>:59: SyntaxWarning: invalid escape sequence '\ '
/tmp/ipykernel_70757/2066166227.py:59: SyntaxWarning: invalid escape sequence '\ '
  print("\ Sources:")


In [32]:
def main_menu():
    while True:
        print("\n" + "=" * 60)
        print("TELECOM EGYPT ASSISTANT")
        print("=" * 60)
        print("1 → Text question")
        print("2 → Voice input")
        print("3 → Document")
        print("Type 'exit' to quit")
        print("=" * 60)
        c = input("Choose (Text/Voice/file): ").strip()

        if c.lower() in ("exit", "quit", "q", "0"):
            print("👋 Goodbye!")
            break

        if c == "text":
            q = input("Your question: ").strip()
            if not q:
                print(" Empty question.")
                continue
            r = process_input(q, input_type='text')
            show_result(r)

        elif c == "voice":
            print("\n1 → Upload audio file")
            print("2 → Record from browser mic")
            vc = input("Choose (1/2): ").strip()

            if vc == "1":
                uploaded = files.upload()
                if uploaded:
                    path = f"/content/{list(uploaded.keys())[0]}"
                    r = process_input(path, input_type='voice')
                    show_result(r)
                else:
                    print(" No file uploaded.")

            elif vc == "2":
                record_voice()
                break
                print("\n After clicking 'Send to Colab', the answer will appear automatically.")

            else:
                print("Invalid choice.")

        elif c == "file":
            print("\n1 → Ingest only (add to KB, no answer)")
            print("2 → Ingest + Query (add to KB, then answer)")
            print("3 → Query only (answer without storing)")
            dc = input("Choose (1/2/3): ").strip()

            if dc not in ("1", "2", "3"):
                print("Invalid choice.")
                continue

            uploaded = files.upload()
            if not uploaded:
                print(" No file uploaded.")
                continue
            path = f"/content/{list(uploaded.keys())[0]}"

            if dc == "1":
                r = process_input(path, input_type='document', doc_action='ingest')
            elif dc == "2":
                q = input("Your question: ").strip()
                if not q:
                    print(" Empty question.")
                    continue
                r = process_input(path, input_type='document',
                                  doc_action='ingest_query', question=q)
            elif dc == "3":
                q = input("Your question: ").strip()
                if not q:
                    print("Empty question.")
                    continue
                r = process_input(path, input_type='document',
                                  doc_action='query', question=q)

            show_result(r)

        else:
            print("Invalid choice. Enter 1, 2, 3, or type 'exit'.")


main_menu()


TELECOM EGYPT ASSISTANT
1 → Text question
2 → Voice input
3 → Document
Type 'exit' to quit
Choose (Text/Voice/file): text
Your question: what is internet packages?

 Answer: Internet packages include:

1. Unlimited internet access for daily use at no cost.
2. Unlimited internet access for one day only with renewal options available.
3. Unlimited internet access for ten Egyptian pounds per month.
4. Unlimited internet access for twenty Egyptian pounds per month.
5. Unlimited internet access for forty Egyptian pounds per month.
6. Unlimited internet access for seventy Egyptian pounds per month.
7. Unlimited internet access for one hundred Egyptian pounds per month.
8. Unlimited internet access for two hundred Egyptian pounds per month.

These packages offer unlimited internet usage and can be renewed monthly or annually.
\ Sources:
  [1] FAQ - Telecom Egypt → https://te.eg/about-te/faq
  [2] باقاتOneHome | خط أرضى مع إنترنت منزلى - Telecom Egypt → https://te.eg/ar/personal/home/one-home

Saving TelecomEgypt_AI_Case_Study.pdf to TelecomEgypt_AI_Case_Study (8).pdf
Your question: what is internet packages?

 Answer: Internet packages include:

1. Unlimited internet access for daily use at no cost.
2. Unlimited internet access for one day only with renewal options available.
3. Unlimited internet access for ten Egyptian pounds per month.
4. Unlimited internet access for twenty Egyptian pounds per month.
5. Unlimited internet access for forty Egyptian pounds per month.
6. Unlimited internet access for seventy Egyptian pounds per month.
7. Unlimited internet access for one hundred Egyptian pounds per month.
8. Unlimited internet access for two hundred Egyptian pounds per month.

These packages offer unlimited internet usage and can be renewed monthly or annually.
\ Sources:
  [1] FAQ - Telecom Egypt → https://te.eg/about-te/faq
  [2] باقاتOneHome | خط أرضى مع إنترنت منزلى - Telecom Egypt → https://te.eg/ar/personal/home/one-home/600min/30mbps
  [3] عرض الصيف من WE GOLD - Tel

 Recording received


/usr/local/lib/python3.13/dist-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


 transcript: 'What is internet packages?'

You said: What is internet packages?
Answer: Internet packages include:

1. Unlimited internet for daily use
2. Unlimited internet for one month
3. Unlimited internet for two months
4. Unlimited internet for three months
5. Unlimited internet for four months
6. Unlimited internet for five months
7. Unlimited internet for six months
8. Unlimited internet for seven months
9. Unlimited internet for eight months
10. Unlimited internet for nine months
11. Unlimited internet for ten months
12. Unlimited internet for eleven months
13. Unlimited internet for twelve months
14. Unlimited internet for thirteen months
15. Unlimited internet for fourteen months
16. Unlimited internet for fifteen months
17. Unlimited internet for sixteen months
18. Unlimited internet for seventeen months
19. Unlimited internet for eighteen months
20. Unlimited internet for nineteen months
21. Unlimited internet for twenty months
22. Unlimited internet for twenty-one months



TELECOM EGYPT ASSISTANT
1 → Text question
2 → Voice input
3 → Document
Type 'exit' to quit
Choose (Text/Voice/file): 

In [34]:
# app_code = '''
# import streamlit as st
# import os
# import tempfile

# # ══════════════════════════════════════════════════════════════
# # BACKEND IMPORT — the methods must be defined before launching
# # ══════════════════════════════════════════════════════════════

# # These are assumed available in the app's Python environment:
# # process_input, handle_text, handle_voice, doc_ingest,
# # doc_ingest_and_query, doc_query_only

# st.set_page_config(page_title="Telecom Egypt Assistant", page_icon="📞", layout="wide")
# st.title("📞 Telecom Egypt Assistant")
# st.caption("Text · Voice · Documents — Arabic / English / Egyptian dialect")


# # ══════════════════════════════════════════════════════════════
# # SIDEBAR — Input mode + Voice/Doc options
# # ══════════════════════════════════════════════════════════════
# with st.sidebar:
#     st.header("⚙️ Mode")

#     mode = st.radio("Input type", ["💬 Text", "🎤 Voice", "📄 Document"])

#     # ─── VOICE OPTIONS ───
#     if mode == "🎤 Voice":
#         st.subheader("Voice input")
#         v_sub = st.radio("Source", ["Upload audio", "Record mic"], key="voice_sub")

#         if v_sub == "Upload audio":
#             audio_file = st.file_uploader("Upload audio", type=["wav", "mp3", "m4a", "webm", "ogg"])
#             if audio_file:
#                 path = os.path.join("/content", audio_file.name)
#                 with open(path, "wb") as f:
#                     f.write(audio_file.getbuffer())
#                 st.audio(path)
#                 if st.button("Ask", key="voice_upload_ask"):
#                     with st.spinner("Transcribing + thinking..."):
#                         r = process_input(path, input_type="voice")
#                         st.session_state.last_result = r

#         else:  # Record mic
#             audio_rec = st.audio_input("Record your question")
#             if audio_rec:
#                 path = "/content/recording_ui.webm"
#                 with open(path, "wb") as f:
#                     f.write(audio_rec.getbuffer())
#                 if st.button("Ask", key="voice_record_ask"):
#                     with st.spinner("Transcribing + thinking..."):
#                         r = process_input(path, input_type="voice")
#                         st.session_state.last_result = r

#     # ─── DOCUMENT OPTIONS ───
#     if mode == "📄 Document":
#         st.subheader("Document")
#         doc_file = st.file_uploader("Upload", type=["pdf", "docx", "txt", "png", "jpg", "jpeg"])
#         doc_action = st.radio("Action", ["Ingest only", "Ingest + Query", "Query only"])
#         doc_q = st.text_input("Question (for Query options)")

#         if doc_file:
#             path = os.path.join("/content", doc_file.name)
#             with open(path, "wb") as f:
#                 f.write(doc_file.getbuffer())
#             st.success(f"Uploaded: {doc_file.name}")

#             if st.button("Run", key="doc_run"):
#                 with st.spinner("Processing..."):
#                     if doc_action == "Ingest only":
#                         r = process_input(path, input_type="document", doc_action="ingest")
#                     elif doc_action == "Ingest + Query":
#                         r = process_input(path, input_type="document",
#                                           doc_action="ingest_query", question=doc_q)
#                     else:
#                         r = process_input(path, input_type="document",
#                                           doc_action="query", question=doc_q)
#                     st.session_state.last_result = r

#     # ─── CLEAR HISTORY BUTTON ───
#     st.divider()
#     if st.button("🗑️ Clear chat history"):
#         st.session_state.messages = []
#         st.rerun()


# # ══════════════════════════════════════════════════════════════
# # CHAT HISTORY
# # ══════════════════════════════════════════════════════════════
# if "messages" not in st.session_state:
#     st.session_state.messages = []

# # Render previous messages
# for msg in st.session_state.messages:
#     with st.chat_message(msg["role"]):
#         st.markdown(msg["content"])
#         if msg.get("citations"):
#             for c in msg["citations"]:
#                 st.markdown(f"- [{c['id']}] {c['title']} → {c['url']}")
#         if msg.get("audio") and os.path.exists(msg["audio"]):
#             st.audio(msg["audio"])


# # ══════════════════════════════════════════════════════════════
# # TEXT INPUT (bottom bar)
# # ══════════════════════════════════════════════════════════════
# if mode == "💬 Text":
#     if prompt := st.chat_input("Type your question..."):
#         # Append user message
#         st.session_state.messages.append({"role": "user", "content": prompt})
#         with st.chat_message("user"):
#             st.markdown(prompt)

#         # Run backend
#         with st.chat_message("assistant"):
#             with st.spinner("Thinking..."):
#                 r = process_input(prompt, input_type="text")
#                 st.markdown(r["answer"])
#                 for c in r.get("citations", []):
#                     st.markdown(f"- [{c['id']}] {c['title']} → {c['url']}")

#                 # Store
#                 st.session_state.messages.append({
#                     "role": "assistant",
#                     "content": r["answer"],
#                     "citations": r.get("citations", [])
#                 })


# # ══════════════════════════════════════════════════════════════
# # SHOW LAST RESULT (from voice / document sidebar)
# # ══════════════════════════════════════════════════════════════
# if "last_result" in st.session_state and mode in ("🎤 Voice", "📄 Document"):
#     r = st.session_state.last_result
#     st.divider()
#     st.subheader("Result")
#     if r.get("transcript"):
#         st.markdown(f"**📝 You said:** {r['transcript']}")
#     st.markdown(f"**🤖 Answer:** {r.get('answer', '')}")
#     for c in r.get("citations", []):
#         st.markdown(f"- [{c['id']}] {c['title']} → {c['url']}")
#     if r.get("audio") and os.path.exists(r["audio"]):
#         st.audio(r["audio"])

#     # Save to chat history
#     st.session_state.messages.append({
#         "role": "assistant",
#         "content": r.get("answer", ""),
#         "citations": r.get("citations", []),
#         "audio": r.get("audio")
#     })
#     del st.session_state.last_result
# '''

# with open('/content/app.py', 'w') as f:
#     f.write(app_code)

In [36]:
# !pip install -q streamlit pyngrok
# print("Installed")

# from pyngrok import ngrok
# import subprocess, time

# NGROK_TOKEN = "2wlVJX3akT7eoeLRHijxjPCLELg_4hTLoAHvX2zPatxBT9bZU"
# ngrok.set_auth_token(NGROK_TOKEN)

# !pkill -f streamlit
# !pkill -f ngrok

# subprocess.Popen(["streamlit", "run", "/content/app.py",
#                   "--server.port", "8501",
#                   "--server.headless", "true"])

# time.sleep(10)
# url = ngrok.connect(8501)
# print(f"\n OPEN THIS URL: {url}\n")

Installed

 OPEN THIS URL: NgrokTunnel: "https://981a-34-106-237-34.ngrok-free.app" -> "http://localhost:8501"



In [39]:
backend_code = '''
import os, re, json, base64, subprocess, pickle
import numpy as np
import faiss
import whisper
import pymupdf
import docx
import pytesseract
from PIL import Image
from gtts import gTTS
from sentence_transformers import SentenceTransformer
from langchain_text_splitters import RecursiveCharacterTextSplitter
from transformers import pipeline

# ─── LLM ───
print("Loading Qwen...")
llm = pipeline(
    "text-generation",
    model="Qwen/Qwen2.5-0.5B-Instruct",
    device=-1,
    max_new_tokens=200,
    do_sample=False,
    temperature=None, top_p=None, top_k=None
)
print("✅ Qwen loaded")

# ─── chunking ───
_text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500, chunk_overlap=50,
    separators=["\\n\\n", "\\n", ".", " ", ""]
)

def chunk_text(text, metadata=None):
    meta = metadata or {}
    pieces = _text_splitter.split_text(text)
    return [
        {'text': p.strip(),
         'source': meta.get('source', 'unknown'),
         'title': meta.get('title', ''),
         'type': meta.get('type', 'text')}
        for p in pieces if len(p.strip()) > 50
    ]

# ─── embeddings ───
_embed_model = None
def get_embed_model():
    global _embed_model
    if _embed_model is None:
        print("Loading embedding model...")
        _embed_model = SentenceTransformer('BAAI/bge-m3')
        print("Model loaded")
    return _embed_model

def embed_texts(texts):
    return get_embed_model().encode(texts, show_progress_bar=False, batch_size=32)

# ─── FAISS ───
_index = None
_chunks = []

def create_index(dimension=1024):
    global _index, _chunks
    _index = faiss.IndexFlatL2(dimension)
    _chunks = []
    print(f"Index created (dim={dimension})")

def add_to_index(chunks):
    global _index, _chunks
    if not chunks:
        return 0
    texts = [c['text'] for c in chunks]
    embeddings = embed_texts(texts).astype('float32')
    if _index is None:
        create_index(embeddings.shape[1])
    _index.add(embeddings)
    _chunks.extend(chunks)
    return len(chunks)

def search_index(query, top_k=3):
    if _index is None or _index.ntotal == 0:
        return []
    query_emb = embed_texts([query]).astype('float32')
    distances, indices = _index.search(query_emb, top_k)
    results = []
    for i, idx in enumerate(indices[0]):
        if idx >= 0:
            results.append({**_chunks[idx], 'distance': float(distances[0][i])})
    return results

def save_index(index_path='index.faiss', chunks_path='chunks.pkl'):
    faiss.write_index(_index, index_path)
    with open(chunks_path, 'wb') as f:
        pickle.dump(_chunks, f)

def load_index(index_path='baseline_index.faiss', chunks_path='baseline_chunks.pkl'):
    global _index, _chunks
    if not os.path.exists(index_path) or not os.path.exists(chunks_path):
        print(f"Index files not found")
        return
    _index = faiss.read_index(index_path)
    with open(chunks_path, 'rb') as f:
        _chunks = pickle.load(f)
    print(f"Loaded: {_index.ntotal} vectors, {len(_chunks)} chunks")

# ─── RAG ───
def generate_with_citations(query, top_k=3):
    results = search_index(query, top_k=top_k + 2)
    results = [r for r in results if 'ir.te.eg' not in r.get('source', '')]
    seen = set(); filtered = []
    for r in results:
        if r['source'] not in seen:
            filtered.append(r); seen.add(r['source'])
        if len(filtered) >= top_k: break
    results = filtered
    if not results:
        return {"answer": "I don't have that information.", "citations": []}
    context = "\\n\\n".join([f"[{i+1}] {r['text'][:400]}" for i, r in enumerate(results)])
    is_ar = any('\\u0600' <= c <= '\\u06FF' for c in query)
    sys_prompt = ("أجب فقط من السياق. استشهد بـ [1]، [2]." if is_ar else
                  "Answer ONLY from the context. Cite using [1], [2].")
    messages = [
        {"role": "system", "content": sys_prompt},
        {"role": "user", "content": f"Context:\\n{context}\\n\\nQuestion: {query}\\n\\nAnswer:"}
    ]
    output = llm(messages, max_new_tokens=200)
    answer = output[0]['generated_text'][-1]['content'].strip()
    citations = [{'id': i+1, 'title': r['title'], 'url': r['source'],
                  'type': r['type'], 'snippet': r['text'][:150]+'...'}
                 for i, r in enumerate(results)]
    return {'answer': answer, 'citations': citations}

# ─── Whisper ───
_whisper_model = None
def get_whisper():
    global _whisper_model
    if _whisper_model is None:
        _whisper_model = whisper.load_model("medium")
    return _whisper_model

def convert_to_wav(input_path, output_path="/content/fixed_audio.wav"):
    if not os.path.exists(input_path): return None
    cmd = ["ffmpeg","-y","-i",input_path,"-ac","1","-ar","16000","-f","wav",output_path]
    try:
        subprocess.run(cmd, check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        return output_path
    except: return None

def voice_to_text(audio_path):
    if not audio_path or not os.path.exists(audio_path):
        return {'transcript':'','success':False}
    wav = convert_to_wav(audio_path)
    if not wav: return {'transcript':'','success':False}
    result = get_whisper().transcribe(wav, language=None)
    t = result['text'].strip()
    return {'transcript': t, 'success': bool(t)}

def text_to_speech(text, output_path="/content/answer.mp3"):
    lang = 'ar' if any('\\u0600' <= c <= '\\u06FF' for c in text) else 'en'
    gTTS(text=text, lang=lang, slow=False).save(output_path)
    return output_path

def handle_voice(audio_path):
    if not audio_path or not os.path.exists(audio_path):
        return {'transcript':'','answer':"Audio not found.",'audio':None,
                'input_type':'voice','citations':[]}
    v = voice_to_text(audio_path)
    if not v['success']:
        return {'transcript':'','answer':"Could not transcribe.",'audio':None,
                'input_type':'voice','citations':[]}
    r = generate_with_citations(v['transcript'])
    try: audio_out = text_to_speech(r.get('answer',''))
    except: audio_out = None
    return {'transcript': v['transcript'], 'answer': r.get('answer',''),
            'audio': audio_out, 'input_type': 'voice',
            'citations': r.get('citations',[])}

# ─── Documents ───
def parse_pdf(path):
    doc = pymupdf.open(path)
    text = "".join(p.get_text()+"\\n" for p in doc); doc.close(); return text
def parse_docx(path): return "\\n".join(p.text for p in docx.Document(path).paragraphs)
def parse_txt(path):
    with open(path,'r',encoding='utf-8',errors='ignore') as f: return f.read()
def parse_image(path): return pytesseract.image_to_string(Image.open(path), lang='ara+eng')

def parse_document(path):
    if not path or not os.path.exists(path): raise FileNotFoundError(path)
    ext = path.lower().rsplit('.',1)[-1]
    if ext=='pdf': return parse_pdf(path)
    if ext=='docx': return parse_docx(path)
    if ext=='txt': return parse_txt(path)
    if ext in ('png','jpg','jpeg'): return parse_image(path)
    raise ValueError(f"Unsupported: .{ext}")

def doc_ingest(path):
    text = parse_document(path)
    if not text.strip():
        return {'answer':"No text extracted.",'citations':[],'input_type':'document'}
    chunks = chunk_text(text, metadata={'source':path,'title':os.path.basename(path),'type':'user_document'})
    n = add_to_index(chunks)
    return {'answer':f"Ingested {n} chunks.",'citations':[],'input_type':'document'}

def doc_ingest_and_query(path, question):
    text = parse_document(path)
    if not text.strip():
        return {'answer':"No text extracted.",'citations':[],'input_type':'document'}
    chunks = chunk_text(text, metadata={'source':path,'title':os.path.basename(path),'type':'user_document'})
    add_to_index(chunks)
    r = generate_with_citations(question)
    return {'answer':r['answer'],'citations':r.get('citations',[]),'input_type':'document'}

def doc_query_only(path, question):
    text = parse_document(path)
    if not text.strip():
        return {'answer':"No text extracted.",'citations':[],'input_type':'document'}
    chunks = chunk_text(text, metadata={'source':path,'title':os.path.basename(path),'type':'temp'})
    q_emb = embed_texts([question]).astype('float32')
    c_embs = embed_texts([c['text'] for c in chunks]).astype('float32')
    temp = faiss.IndexFlatL2(c_embs.shape[1]); temp.add(c_embs)
    d, idxs = temp.search(q_emb, min(3, len(chunks)))
    results = [{**chunks[i], 'distance': float(d[0][j])} for j,i in enumerate(idxs[0]) if i>=0]
    context = "\\n".join(r['text'][:300] for r in results)
    msgs = [{"role":"system","content":"Answer ONLY from context."},
            {"role":"user","content":f"Context:\\n{context}\\n\\nQuestion: {question}"}]
    out = llm(msgs, max_new_tokens=200)
    ans = out[0]['generated_text'][-1]['content'].strip()
    cites = [{'id':i+1,'title':r['title'],'url':r['source'],'type':r['type']}
             for i,r in enumerate(results)]
    return {'answer':ans,'citations':cites,'input_type':'document'}

# ─── Text ───
def handle_text(query):
    if not query or not query.strip():
        return {'answer':"Please provide a question.",'citations':[],'audio':None}
    r = generate_with_citations(query.strip())
    return {'answer':r['answer'],'citations':r['citations'],'audio':None,'input_type':'text'}

# ─── Router ───
def process_input(user_input=None, input_type=None, doc_action=None, question=None):
    if input_type is None:
        if user_input is None:
            return {'answer':"No input.",'input_type':'unknown'}
        ext = user_input.lower().rsplit('.',1)[-1] if '.' in user_input else ''
        if ext in ('wav','mp3','m4a','webm','ogg'): input_type='voice'
        elif ext in ('pdf','docx','txt','png','jpg','jpeg'): input_type='document'
        else: input_type='text'
    if input_type=='text': return handle_text(user_input)
    if input_type=='voice': return handle_voice(user_input)
    if input_type=='document':
        action = doc_action or 'ingest'
        if action=='ingest': return doc_ingest(user_input)
        if action=='ingest_query': return doc_ingest_and_query(user_input, question or "")
        if action=='query': return doc_query_only(user_input, question or "")
    return {'answer':"Unknown.",'input_type':'unknown'}

# ─── auto-load baseline index ───
load_index('baseline_index.faiss', 'baseline_chunks.pkl')
'''

with open('/content/backend.py', 'w') as f:
    f.write(backend_code)

print("backend.py written")

backend.py written


In [40]:
with open('/content/app.py', 'r') as f:
    app_code = f.read()

old = """# These are assumed available in the app's Python environment:
# process_input, handle_text, handle_voice, doc_ingest,
# doc_ingest_and_query, doc_query_only"""

new = """from backend import (
    process_input,
    handle_text, handle_voice,
    doc_ingest, doc_ingest_and_query, doc_query_only,
)"""

app_code = app_code.replace(old, new)

with open('/content/app.py', 'w') as f:
    f.write(app_code)

print(" app.py patched")

 app.py patched


In [41]:
!pkill -f streamlit
!pkill -f ngrok

import subprocess, time
subprocess.Popen(["streamlit", "run", "/content/app.py",
                  "--server.port", "8501",
                  "--server.headless", "true"])

time.sleep(15)   # longer — models take time to load

from pyngrok import ngrok
url = ngrok.connect(8501)
print(f"\nOPEN THIS URL: {url}\n")


OPEN THIS URL: NgrokTunnel: "https://1fc7-34-106-237-34.ngrok-free.app" -> "http://localhost:8501"

